In [1]:
# Loading data from system and loading tool box
import sys
sys.path.append("../src")

import pandas as pd

from gow_rag.data_loader import load_corpus, load_claims
from gow_rag.preprocessing import make_tokenizer
from gow_rag.sparse_retrieval import SparseRetriever, count_words
from gow_rag.graph_of_words import graph_term_weights
from gow_rag.evaluation import evaluate, evaluate_per_query, paired_t_test
from gow_rag.paths import TABLES_DIR

tokenizer = make_tokenizer("english", stem=True)   # the paper's text preparation

corpus = load_corpus()
claim_sets = {split: load_claims(split) for split in ["validation", "train"]}

print("Papers:", len(corpus))
for split, (queries, qrels) in claim_sets.items():
    print(f"{split} claims:", len(queries))

Papers: 5183
validation claims: 188
train claims: 505


In [2]:
# Building 4 search engines
doc_ids = corpus["doc_id"].tolist()
doc_tokens = [tokenizer(text) for text in corpus["full_text"]]
doc_lengths = [len(tokens) for tokens in doc_tokens]
doc_counts = [count_words(tokens) for tokens in doc_tokens]
doc_graph = [graph_term_weights(tokens) for tokens in doc_tokens]

engines = {
    "TF-IDF": SparseRetriever(doc_ids, doc_counts, doc_lengths, model="tfidf"),
    "BM25": SparseRetriever(doc_ids, doc_counts, doc_lengths, model="bm25"),
    "TW-IDF": SparseRetriever(doc_ids, doc_graph, doc_lengths, model="twidf"),
    "TW-IDF-log": SparseRetriever(doc_ids, doc_graph, doc_lengths, model="twidf_log"),
}
print("Ready:", list(engines))

Ready: ['TF-IDF', 'BM25', 'TW-IDF', 'TW-IDF-log']


In [3]:
#testing with claims dataset
per_query = {}   
rows = []
for split, (queries, qrels) in claim_sets.items():
    per_query[split] = {}
    for name, engine in engines.items():
        run = engine.search_all(queries, tokenizer)
        per_query[split][name] = evaluate_per_query(run, qrels)
        rows.append({"Claims": split, "Method": name, **evaluate(run, qrels)})

main_results = pd.DataFrame(rows).set_index(["Claims", "Method"])
main_results

MAP    P@10  Recall@10  MRR@10  nDCG@10
Claims     Method                                                
validation TF-IDF      0.7822  0.1000     0.9246  0.7849   0.8171
           BM25        0.7809  0.0995     0.9235  0.7836   0.8159
           TW-IDF      0.6036  0.0883     0.8335  0.6069   0.6580
           TW-IDF-log  0.7574  0.0973     0.9098  0.7601   0.7939
train      TF-IDF      0.8118  0.1026     0.9322  0.8161   0.8413
           BM25        0.8192  0.1024     0.9307  0.8235   0.8465
           TW-IDF      0.6553  0.0950     0.8715  0.6593   0.7069
           TW-IDF-log  0.8099  0.1010     0.9182  0.8133   0.8358

In [4]:
comparisons = [("TW-IDF", "BM25"), ("TW-IDF-log", "BM25"), ("TW-IDF-log", "TW-IDF")]

test_rows = []
for split in claim_sets:
    for a, b in comparisons:
        result = paired_t_test(per_query[split][a], per_query[split][b], metric="AP")
        test_rows.append({
            "Claims": split,
            "Comparison": f"{a} vs {b}",
            **result,
            "Significant (p < 0.05)?": "yes" if result["p_value"] < 0.05 else "no",
        })

significance = pd.DataFrame(test_rows)
significance["p_value"] = significance["p_value"].map(lambda p: f"{p:.2g}")
significance

,Claims,Comparison,mean_difference,p_value,Significant (p < 0.05)?
0,validation,TW-IDF vs BM25,-0.1773,7.3e-11,yes
1,validation,TW-IDF-log vs BM25,-0.0235,0.017,yes
2,validation,TW-IDF-log vs TW-IDF,0.1538,5.2e-09,yes
3,train,TW-IDF vs BM25,-0.1640,1.4e-26,yes
4,train,TW-IDF-log vs BM25,-0.0093,0.12,no
5,train,TW-IDF-log vs TW-IDF,0.1546,1.4e-23,yes


In [5]:
#checking if the claim length matter
rows = []
for split, (queries, qrels) in claim_sets.items():
    for claim_id, text in queries.items():
        row = {"length": len(set(tokenizer(text)))}
        for name in engines:
            row[name] = per_query[split][name].loc[claim_id, "AP"]
        rows.append(row)
by_claim = pd.DataFrame(rows)

by_claim["Claim length (words)"] = pd.cut(by_claim["length"], [0, 5, 8, 11, 100],
                                          labels=["1-5", "6-8", "9-11", "12+"])
length_table = by_claim.groupby("Claim length (words)", observed=True).agg(
    claims=("BM25", "size"),
    BM25=("BM25", "mean"),
    **{"TW-IDF": ("TW-IDF", "mean"), "TW-IDF-log": ("TW-IDF-log", "mean")},
).round(3)
length_table["Gap: BM25 - TW-IDF"] = (length_table["BM25"] - length_table["TW-IDF"]).round(3)
length_table

,claims,BM25,TW-IDF,TW-IDF-log,Gap: BM25 - TW-IDF
Claim length (words),,,,,
1-5,104,0.773,0.543,0.766,0.230
6-8,290,0.785,0.589,0.772,0.196
9-11,188,0.827,0.695,0.807,0.132
12+,111,0.873,0.779,0.865,0.094


In [6]:
#saving results
main_results.to_csv(TABLES_DIR / "main_results.csv")
significance.to_csv(TABLES_DIR / "significance_tests.csv", index=False)
length_table.to_csv(TABLES_DIR / "results_by_claim_length.csv")

for split in per_query:
    scores = pd.concat(per_query[split], names=["Method"]).reset_index()
    scores.to_csv(TABLES_DIR / f"per_query_scores_{split}.csv", index=False)

print("Saved 5 tables in:", TABLES_DIR)

Saved 5 tables in: C:\Users\Lenovo\Desktop\Projects for IP Paris Application\Graph of words (Reviewing the paper with dataset)\github_upload\scientific-literature-GOW-RAG\results\tables
